# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My Lane:** Lane 2 - User Engagement & CTR Prediction

**ML Task Type:** Classification

**Why Classification?**
I want to predict whether a user will click on a search result (click = 1, no click = 0). This is a binary outcome. The model will assign a probability score between 0 and 1, and we can threshold it to make a yes/no decision.

**Alternative Task Types Considered:**
- **Ranking/Scoring:** Could also work (ranking pages by CTR), but Classification fits better because the action is "fix this page or not" (a binary decision)

**The Decision This Improves:**
Which pages should a content creator prioritize for improvement? A classification model tells them: "these pages are likely to underperform — fix them first."

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** Click-Through Rate (CTR) — whether a user clicks on a search result

**What I'll Predict:**
A probability score between 0 and 1 representing the likelihood of a click

**Is the target observed or defined?**
Observed! CTR is measured from actual user behavior (clicks / impressions). This is a real outcome, not a rule someone made up.

**How I'll create the target column:**
`clicked = 1 if CTR > median(CTR) else 0`

**What the target column looks like:**

| avg_position | ctr | clicked (target) |
|--------------|-----|------------------|
| 1 | 0.12 | 1 |
| 3 | 0.04 | 0 |
| 2 | 0.08 | 1 |
| 5 | 0.01 | 0 |

**Why this target makes sense:**
- It's directly tied to the decision (fix pages with low predicted CTR)
- It's observed in the data (not defined by a rule)
- It's measurable and actionable

In [ ]:
# Create the target column from the data
import pandas as pd
import numpy as np

df = pd.read_csv('data/sample_data.csv')
print("Data loaded!")
print(f"Shape: {df.shape}")

# Create target: clicked = 1 if CTR is above median
median_ctr = df['ctr'].median()
df['clicked'] = (df['ctr'] > median_ctr).astype(int)

print(f"\nMedian CTR: {median_ctr:.3f}")
print(f"Target distribution:")
print(df['clicked'].value_counts())
print(f"\nClick rate: {df['clicked'].mean():.3f}")

# Show what the target column looks like
print("\nSample of data with target column:")
print(df[['avg_position', 'ctr', 'clicked']].head(10))

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** Precision@50

**Why Precision@50?**
- The action is: "fix the top 50 pages that need improvement"
- We care about the precision of our top 50 predictions
- False positives = wasted effort on pages that don't need fixing
- This matches the real-world decision

**Definition:**
> Precision@50 = (Number of truly clicked pages in top 50 predictions) / 50

**What number means 'good'?**
- Baseline (position-only rule): ~0.65-0.70
- Good ML model: >0.75
- Excellent: >0.80

**Secondary Metrics:**
- **ROC-AUC:** Overall ranking ability (does the model separate good from bad?)
- **Recall:** Are we catching the pages that truly need fixing?

In [ ]:
# Show how we'd calculate Precision@50
from sklearn.metrics import precision_score

# Simulate predictions (using position as a simple proxy)
df['predicted_click'] = (df['avg_position'] < 3).astype(int)

# Calculate Precision@50 (top 50 predictions)
top50_indices = df.nlargest(50, 'predicted_click').index
y_true_top50 = df.loc[top50_indices, 'clicked']
y_pred_top50 = df.loc[top50_indices, 'predicted_click']

precision_at_50 = precision_score(y_true_top50, y_pred_top50)
print(f"Precision@50 (using position as predictor): {precision_at_50:.3f}")
print(f"\nThis is our baseline. The ML model should beat this!")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = one page/URL with its search performance metrics

**What this means:**
Each row represents a unique page and its performance in search results. We're predicting the behavior of each page individually.

**Why this unit makes sense:**
- The decision is: "which pages to fix" → we need per-page predictions
- Content creators think in terms of pages, not impressions
- The action applies at the page level

In [ ]:
# Show the unit of analysis as a real dataframe
print("Unit of Analysis: One row = one page")
print("="*60)

# Select a sample of columns that show the page-level view
unit_df = df[['avg_position', 'impressions_90d', 'ctr', 'clicked']].copy()

# Add a page_id (just for demonstration)
unit_df['page_id'] = range(1, len(unit_df) + 1)
unit_df = unit_df[['page_id', 'avg_position', 'impressions_90d', 'ctr', 'clicked']]

print("Sample of the data (one row = one page):")
print(unit_df.head(15))

print(f"\nTotal pages: {len(unit_df)}")
print(f"Each row is one page with its performance metrics")
print(f"The target column 'clicked' indicates if CTR is above median")

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why a simple rule isn't enough:**

**Rule-based approach:** "If position < 3, predict click"

**Problems with this rule:**
1. Position alone is noisy — some position 1 pages have low CTR
2. Other factors matter: content type, freshness, impressions
3. The relationship is non-linear and interactions exist
4. The optimal rule changes over time

**Why ML helps:**
1. **Multiple signals:** ML can combine position, content type, freshness, and other features
2. **Interactions:** ML can learn that freshness matters more for some content types
3. **Adaptability:** ML models can retrain as patterns change
4. **Measurable improvement:** We can test if ML beats the rule on held-out data

**What a fixed rule would miss:**
- A page in position 2 with fresh content might outperform position 1 with old content
- Content type matters: videos might need different rules than articles
- The relationship between features is complex and data-dependent

In [ ]:
# Show why a rule isn't enough
print("Why position alone is noisy:")
print("="*60)

# Group by position and show CTR spread
position_stats = df.groupby('avg_position')['ctr'].agg(['mean', 'std', 'min', 'max'])
print(position_stats.head(10))

print("\nEven at position 1, CTR varies widely!")
print("This means position alone is not enough to predict clicks reliably.")
print("ML can combine other signals to make better predictions.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**My repo URL:** https://github.com/noor-meer/flyrank-ml-internship

**File location:** work/notebooks/w02_ml_task_framing.ipynb